# 5. Sprzątanie środowiska warsztatowego

Notebook usuwa zarejestrowany kernel `Supertonic Workshop (.venv)` i katalog `302-tts-supertonic/.venv`.

> Uruchom go z bazowego kernela Python, nie z kernela znajdującego się w usuwanym `.venv`. Pliki WAV, wejściowe JSON-y i cache modelu Supertonic nie są usuwane.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

def find_project_dir() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, current / "302-tts-supertonic", *current.parents]:
        if (candidate / "requirements.txt").is_file() and (candidate / "webgui").is_dir():
            return candidate
    raise FileNotFoundError("Nie znaleziono katalogu 302-tts-supertonic.")

PROJECT_DIR = find_project_dir()
VENV_DIR = (PROJECT_DIR / ".venv").resolve()
KERNEL_NAME = "supertonic-workshop"

print("Do usunięcia:", VENV_DIR)
print("Aktywny interpreter:", sys.executable)

## Kontrola bezpieczeństwa

Ustaw `CONFIRM_DELETE_VENV = True` dopiero po sprawdzeniu wyświetlonej ścieżki.

In [ ]:
CONFIRM_DELETE_VENV = False

if Path(sys.prefix).resolve() == VENV_DIR:
    raise RuntimeError(
        "Ten notebook działa z usuwanego .venv. Przełącz kernel na bazowy Python, "
        "uruchom kernel ponownie i wróć do tej komórki."
    )

if VENV_DIR.parent != PROJECT_DIR or VENV_DIR.name != ".venv":
    raise RuntimeError(f"Odmowa usunięcia nieoczekiwanej ścieżki: {VENV_DIR}")

if not CONFIRM_DELETE_VENV:
    raise RuntimeError("Najpierw ustaw CONFIRM_DELETE_VENV = True.")

print("Kontrole zakończone. Można wykonać następną komórkę.")

In [ ]:
venv_python = VENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if venv_python.is_file():
    subprocess.run(
        [str(venv_python), "-m", "jupyter", "kernelspec", "remove", "-f", KERNEL_NAME],
        check=False,
    )

if VENV_DIR.is_dir():
    shutil.rmtree(VENV_DIR)
    print("Usunięto:", VENV_DIR)
else:
    print("Brak .venv — nic do usunięcia.")

## Co pozostaje na dysku?

- `outputs/` — wyniki ćwiczeń z notebooków,
- `generated_audio/` — pliki utworzone przez endpoint serwera,
- cache modelu Supertonic w profilu użytkownika.

Te dane można przejrzeć i usunąć osobno. Notebook celowo ich nie kasuje.